# vbgr2 — Colab runner

Same pipeline as the CLI, wrapped for Colab. Run the cells in order.

**Before you start:** Runtime → Change runtime type → GPU (A100 or L4).
SAM2Matting with the SAM3 backbone wants ~24 GB for 1080p; on a 16 GB card
set `cfg.io.max_size = 720`, and on anything smaller use the SAM2.1-B+
checkpoint instead.

**Licence reminder:** SAM2Matting is CC BY-NC-SA 4.0 and MatAnyone 2 is NTU
S-Lab — both non-commercial. `docs/LICENSING.md` has the details and the
options. Ultralytics YOLOv8 is AGPL-3.0, which matters even for the research
path.

## 1. Environment check

Stops early rather than failing 20 minutes into a download.

In [ ]:
import subprocess, sys
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total',
                      '--format=csv,noheader'],
                     capture_output=True, text=True).stdout or 'NO GPU')
print(sys.version)

## 2. Install

In [ ]:
%%bash
set -e
pip -q install opencv-python-headless scipy PyYAML ultralytics transformers iopath imageio imageio-ffmpeg

# vbgr2 itself: upload the repo, or clone it if you have pushed it somewhere.
if [ ! -d vbgr2 ]; then
  echo 'Upload the vbgr2 folder to /content (or git clone it) before continuing.'
fi

mkdir -p third_party
[ -d third_party/SAM2Matting ] || git clone -q https://github.com/FudanCVL/SAM2Matting third_party/SAM2Matting
pip -q install -e third_party/SAM2Matting

[ -d third_party/MatAnyone2 ] || git clone -q https://github.com/pq-yang/MatAnyone2 third_party/MatAnyone2
pip -q install -e third_party/MatAnyone2
echo done

## 3. Checkpoints

SAM2Matting ships three variants. `SAM2Matting-SAM3.pt` is the one that
targets rapid motion and supports text prompts.

In [ ]:
%%bash
mkdir -p checkpoints pretrained_models
pip -q install huggingface_hub
python - <<'PY'
from huggingface_hub import hf_hub_download
import shutil
p = hf_hub_download('FudanCVL/SAM2Matting', 'checkpoints/SAM2Matting-SAM3.pt')
shutil.copy(p, 'checkpoints/SAM2Matting-SAM3.pt')
print('sam2matting ok')
PY
# MatAnyone 2 downloads its own checkpoint on first inference.
wget -q -nc https://github.com/pq-yang/MatAnyone2/releases/download/v1.0.0/matanyone2.pth -P pretrained_models/ || true
ls -la checkpoints pretrained_models

## 4. Selftest — always run this before a batch

CPU only, ~15 s, no models. If it fails, the problem is the install, not the footage.

In [ ]:
import sys; sys.path.insert(0, '/content/vbgr2')
!cd /content/vbgr2 && python -m vbgr.cli selftest

In [ ]:
!cd /content/vbgr2 && python -m vbgr.cli engines

## 5. Configure

In [ ]:
from vbgr.config import Config

cfg = Config()
cfg.io.input_dir  = '/content/clips'
cfg.io.output_dir = '/content/results'
cfg.io.output_mode = 'all'          # green + alpha pass + transparent webm
cfg.io.max_size = None              # set 720 on a 16 GB card

cfg.matting.engine = 'sam2matting'  # or 'matanyone2', or 'rvm' for the baseline
cfg.matting.checkpoint = '/content/checkpoints/SAM2Matting-SAM3.pt'

# The three v2 additions. Turn them off one at a time to attribute a change.
cfg.motion.enabled      = True      # flow-adaptive trimap + warped alpha prior
cfg.reid.enabled        = True      # DINOv3 re-entry recovery
cfg.memory_gate.enabled = True      # stop bad frames entering memory
cfg.decontam.enabled    = True      # kill the composite fringe

cfg.save('/content/my_config.yaml')
print(open('/content/my_config.yaml').read()[:800])

## 6. Run one clip first

Don't start a batch until one clip looks right.

In [ ]:
from vbgr.pipeline import Pipeline

pipe = Pipeline(cfg)
rep = pipe.run_clip('/content/clips/ipman.mp4')
print(rep.summary())
for s in rep.shots:
    print(f'  shot [{s.start}:{s.end}] kept={s.n_kept} dropped={s.n_dropped} '
          f'gate_rejects={s.gate_rejects} reseeds={s.reseeds} '
          f'reentries={s.reentries} high_motion={s.high_motion_frames}')
    for n in s.notes:
        print('    -', n)
rep.outputs

### Read the shot report before you look at the video

- `gate_rejects` high → the tracker is struggling; look at where.
- `reseeds` > 0 → the track was lost and recovered. Check the stitch points.
- `reentries` > 0 → someone was recovered who the forward pass missed. This is
  the `1917` fix firing.
- `high_motion` ≈ 0 on a fight scene → `motion.high_motion_px` is set too high
  and the blur path never ran.

## 7. Look at the worst frames, not random ones

In [ ]:
!cd /content/vbgr2 && python bench/contact_sheet.py sheet \
    --source /content/results --clip ipman \
    --pattern '{clip}_alpha.mp4' --alpha-from grey \
    --source-pattern '/content/clips/{clip}.mp4' \
    --n 12 --out /content/sheets/ipman.png

from IPython.display import Image
Image('/content/sheets/ipman.png')

## 8. Batch

In [ ]:
from vbgr.pipeline import run_batch
reports = run_batch(cfg)
for r in reports:
    print(r.summary())

## 9. Benchmark: prove it got better

This is the part that was missing before. Score the new run, score the old
outputs, diff them. Both runs must use the same `--short-side`.

In [ ]:
%%bash
cd /content/vbgr2
# old outputs (green composites, no alpha channel)
python bench/run_bench.py score --source '/content/v1_outputs' --run v1 \
    --alpha-from green --short-side 400 --out bench/results/v1.json
# new outputs (real alpha pass)
python bench/run_bench.py score --source /content/results --run v2 \
    --pattern '{clip}_alpha.mp4' --alpha-from grey --short-side 400 \
    --source-pattern '/content/clips/{clip}.mp4' --out bench/results/v2.json
python bench/run_bench.py compare bench/results/v1.json bench/results/v2.json

## 10. Ablate

One switch at a time, same clips, same resolution. Anything that does not move
a number is not worth the complexity it costs.

In [ ]:
import copy, json
from vbgr.pipeline import Pipeline
from bench import metrics as M
from bench.run_bench import load_alphas, load_frames

CLIPS = ['/content/clips/ipman.mp4', '/content/clips/butter.mp4',
         '/content/clips/1917.mp4']

ABLATIONS = {
    'full':        {},
    'no_motion':   {'motion.enabled': False},
    'no_reid':     {'reid.enabled': False},
    'no_gate':     {'memory_gate.enabled': False},
    'no_decontam': {'decontam.enabled': False},
}

def apply(c, overrides):
    c = copy.deepcopy(c)
    for k, v in overrides.items():
        sec, attr = k.split('.')
        setattr(getattr(c, sec), attr, v)
    return c

results = {}
for name, ov in ABLATIONS.items():
    c = apply(cfg, ov)
    c.io.output_dir = f'/content/abl/{name}'
    rows = []
    for p in CLIPS:
        rep = Pipeline(c).run_clip(p, c.io.output_dir)
        al = load_alphas(rep.outputs['alpha'], 'grey', short_side=400)
        fr = load_frames(p, short_side=400)
        n = min(len(al), len(fr))
        rows.append(M.evaluate(rep.name, name, al[:n], fr[:n], stride=4).row())
    results[name] = rows
    print(name, json.dumps(rows))

json.dump(results, open('/content/ablations.json', 'w'), indent=2)